# Lecture 10: Motion Estimation

> https://medium.com/@itberrios6/introduction-to-motion-detection-part-1-e031b0bb9bb2

We implement three methods **Frame Differencing**, **Optical Flow**, und **Background Subtraction** and use them for analyzing a TikTok video of a political speech.

---

## Frame Differencing

The simplest form of motion detection: We subtract two consecutive frames from each other pixel by pixel. Where the brightness has changed significantly, there was motion.

In [7]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from glob import glob
from PIL import Image

In [8]:
VIDEO_PATH = "/Users/clarafochler/Documents/Dissertation/Data/robin_wagener_part3_47insteadof48_gruene/7358388148751142177.mp4"
OUTPUT_DIR = "/Users/clarafochler/Documents/Dissertation/Data/temp_speech"
GIF_PATH   = "/Users/clarafochler/Documents/Dissertation/Data/temp_speech/motion_speech.gif"
MP4_PATH   = "/Users/clarafochler/Documents/Dissertation/Data/temp_speech/motion_speech.mp4"

### Parameter

Was passiert wenn wir `BBOX_THRESH` stark erhöhen (z.B. auf 2000)? Was wenn wir ihn auf 50 setzt?


In [ ]:
# Paramter
BBOX_THRESH  = 200      # [px²] minimal area of a Bounding Box
NMS_THRESH   = 1e-3    # IoU threshold for Non-Maximum Suppression
FRAME_SKIP   = 1        # Evaluate every nth frame (1 = every frame)
MAX_FRAMES   = None     # None = entire video

In [10]:

def get_mask(frame1, frame2, kernel=np.ones((9, 9), dtype=np.uint8)):
    # Measures a binary motion mask between two grayscale frames - white shows motion, black shows no motion
    frame_diff = cv2.subtract(frame2, frame1)
    frame_diff = cv2.medianBlur(frame_diff, 3)

    mask = cv2.adaptiveThreshold(
        frame_diff, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 11, 3
    )
    mask = cv2.medianBlur(mask, 3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    return mask


def get_contour_detections(mask, thresh=200):
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_TC89_L1
    ) # retrieves contours of connected components
    detections = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)  # returns bounding boxes
        area = w * h
        if area > thresh:
            detections.append([x, y, x + w, y + h, area])
    return np.array(detections)


def remove_contained_bboxes(boxes):
    # Removes smaller boxes that are fully contained within larger ones
    check_array = np.array([True, True, False, False]) #[x_min, y_min, x_max, y_max] 
    keep = list(range(len(boxes)))
    for i in keep:
        for j in range(len(boxes)):
            if np.all((np.array(boxes[j]) >= np.array(boxes[i])) == check_array):
                try:
                    keep.remove(j)
                except ValueError:
                    continue
    return keep


def non_max_suppression(boxes, scores, threshold=1e-3):
    # Non-maximum suppression based on IoU
    boxes = boxes[np.argsort(scores)[::-1]]
    order = remove_contained_bboxes(boxes)
    keep = []
    while order:
        i = order.pop(0)
        keep.append(i)
        for j in order[:]:
            intersection = (
                max(0, min(boxes[i][2], boxes[j][2]) - max(boxes[i][0], boxes[j][0]))
                * max(0, min(boxes[i][3], boxes[j][3]) - max(boxes[i][1], boxes[j][1]))
            )
            union = (
                (boxes[i][2] - boxes[i][0]) * (boxes[i][3] - boxes[i][1])
                + (boxes[j][2] - boxes[j][0]) * (boxes[j][3] - boxes[j][1])
                - intersection
            )
            if union > 0 and intersection / union > threshold:
                order.remove(j)
    return boxes[keep]


def get_detections(frame1, frame2,
                   bbox_thresh=200,
                   nms_thresh=1e-3,
                   mask_kernel=np.ones((9, 9), dtype=np.uint8)):
    # Main function: returns NMS-filtered bounding boxes
    mask = get_mask(frame1, frame2, mask_kernel)
    detections = get_contour_detections(mask, bbox_thresh)
    if len(detections) == 0:
        return np.array([])
    bboxes = detections[:, :4]
    scores = detections[:, -1]
    return non_max_suppression(bboxes, scores, nms_thresh)


def draw_bboxes(frame, detections):
    # Draws bounding boxes on the frame
    for det in detections:
        x1, y1, x2, y2 = det.astype(int)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)


def create_gif_from_images(save_path, image_path, ext):
    ext = ext.replace(".", "")
    image_paths = sorted(glob(os.path.join(image_path, f"*.{ext}")))
    image_paths.sort(key=lambda f: int("".join(filter(str.isdigit, f))))
    pil_images = [Image.open(p) for p in image_paths]
    pil_images[0].save(
        save_path, format="GIF",
        append_images=pil_images[1:],
        save_all=True, duration=50, loop=0
    )
    print(f"GIF saved: {save_path}")


### Video verarbeiten

In [11]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH) # opens video and creates VideoCapture object (extracts sequences of frames)
if not cap.isOpened():
    raise FileNotFoundError(f"Video nicht gefunden: {VIDEO_PATH}")

fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {width}x{height} @ {fps:.1f} fps, {total} Frames")

# Read first frame and covert to grayscale
ret, prev_bgr = cap.read()
if not ret:
    raise RuntimeError("First frame error")
prev_gray = cv2.cvtColor(prev_bgr, cv2.COLOR_BGR2GRAY)

frame_idx   = 0
saved_idx   = 0
kernel      = np.ones((9, 9), dtype=np.uint8)

while True:
    ret, curr_bgr = cap.read()
    if not ret:
        break
    frame_idx += 1

    if MAX_FRAMES and frame_idx > MAX_FRAMES:
        break
    if frame_idx % FRAME_SKIP != 0:
        continue

    curr_gray = cv2.cvtColor(curr_bgr, cv2.COLOR_BGR2GRAY)

    detections = get_detections(
        prev_gray, curr_gray,
        bbox_thresh=BBOX_THRESH,
        nms_thresh=NMS_THRESH,
        mask_kernel=kernel
    )

    display = curr_bgr.copy()
    if len(detections) > 0:
        draw_bboxes(display, detections)

    # Frame als PNG für GIF speichern
    out_path = os.path.join(OUTPUT_DIR, f"frame_{saved_idx:05d}.png")
    cv2.imwrite(out_path, display)
    saved_idx += 1

    prev_gray = curr_gray

cap.release()
print(f"{saved_idx} Frames verarbeitet.")

Video: 576x1024 @ 30.0 fps, 2591 Frames
2590 Frames verarbeitet.


In [12]:
# Let's have a look at the results
create_gif_from_images(GIF_PATH, OUTPUT_DIR, ".png")

GIF saved: /Users/clarafochler/Documents/Dissertation/Data/temp_speech/motion_speech.gif


Let's have a look at the output GIF
- Which motion was detected, which was not?
- Where can you see false motion detection?


---

## Optical Flow

We use a dense Optical Flow algorithm **Farneback-Algorithmus**, which  calcualtes for each pixel a motion vector


In [13]:
OUTPUT_DIR = "output_optflow"
GIF_PATH   = "output_optflow/optical_flow_speech.gif"
MP4_PATH   = "output_optflow/optical_flow_speech.mp4"

### Parameter

`MOTION_THRESH` is a range of values – a lower threshold is set at the bottom of the image and a higher threshold at the top. We can experiment with the difference.


In [14]:
MOTION_THRESH  = (0.3, 1.0)  # (min, max) for vertical threshold gradients
                               # What happens with (0.5, 0.5) or (1.0, 0.3)?

### Kernfunktionen Optical Flow

`compute_flow()` returns for each pixel a 2D-Vektor (how far has the pixel moved in x and y direction)

In [15]:
def compute_flow(frame1, frame2):
    # measures dense optical flow (Farneback) between two BGR frames
    gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
    gray1 = cv2.GaussianBlur(gray1, dst=None, ksize=(3, 3), sigmaX=5)
    gray2 = cv2.GaussianBlur(gray2, dst=None, ksize=(3, 3), sigmaX=5)
    flow = cv2.calcOpticalFlowFarneback(
        gray1, gray2, None,
        pyr_scale=0.75, levels=3, winsize=5,
        iterations=3, poly_n=10, poly_sigma=1.2, flags=0
    )
    return flow


def get_flow_viz(flow):
    # Returns a BGR image visualizing the optical flow (we do not use this later on, but it can be helfpul for debugging)
    hsv = np.zeros((flow.shape[0], flow.shape[1], 3), dtype=np.uint8)
    hsv[..., 1] = 255
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


def get_motion_mask(flow_mag, motion_thresh=1, kernel=np.ones((7, 7))):
    # Creates a binary motion mask from the flow magnitude
    motion_mask = np.uint8(flow_mag > motion_thresh) * 255
    motion_mask = cv2.erode(motion_mask, kernel, iterations=1)
    motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    return motion_mask


def get_contour_detections(mask, ang, angle_thresh=2, thresh=200):
    # Finds bounding boxes from contours in the motion mask, using angle consistency
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_TC89_L1
    )
    temp_mask = np.zeros_like(mask)
    angle_thresh = angle_thresh * ang.std()
    detections = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        area = w * h
        cv2.drawContours(temp_mask, [cnt], 0, (255,), -1)
        flow_angle = ang[np.nonzero(temp_mask)]
        if area > thresh and flow_angle.std() < angle_thresh:
            detections.append([x, y, x + w, y + h, area])
    return np.array(detections)


def remove_contained_bboxes(boxes):
    # Removes smaller boxes that are fully contained within larger ones
    check_array = np.array([True, True, False, False])
    keep = list(range(len(boxes)))
    for i in keep:
        for j in range(len(boxes)):
            if np.all((np.array(boxes[j]) >= np.array(boxes[i])) == check_array):
                try:
                    keep.remove(j)
                except ValueError:
                    continue
    return keep


def non_max_suppression(boxes, scores, threshold=1e-3):
    # measures non-maximum suppression based on IoU
    boxes = boxes[np.argsort(scores)[::-1]]
    order = remove_contained_bboxes(boxes)
    keep = []
    while order:
        i = order.pop(0)
        keep.append(i)
        for j in order[:]:
            intersection = (
                max(0, min(boxes[i][2], boxes[j][2]) - max(boxes[i][0], boxes[j][0]))
                * max(0, min(boxes[i][3], boxes[j][3]) - max(boxes[i][1], boxes[j][1]))
            )
            union = (
                (boxes[i][2] - boxes[i][0]) * (boxes[i][3] - boxes[i][1])
                + (boxes[j][2] - boxes[j][0]) * (boxes[j][3] - boxes[j][1])
                - intersection
            )
            if union > 0 and intersection / union > threshold:
                order.remove(j)
    return boxes[keep]


def build_motion_thresh(height, width, thresh_range=(0.3, 1.0)):
    # Builds the vertical threshold gradient for the actual resolution.
    return np.c_[np.linspace(thresh_range[0], thresh_range[1], height)].repeat(width, axis=-1)


def get_detections(frame1, frame2, motion_thresh, ang_unused=None,
                   bbox_thresh=200, nms_thresh=1e-3,
                   mask_kernel=np.ones((7, 7), dtype=np.uint8)):
    # Main function: Optical Flow → Mask → Bounding Boxes (NMS-filtered)
    flow = compute_flow(frame1, frame2)
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    motion_mask = get_motion_mask(mag, motion_thresh=motion_thresh, kernel=mask_kernel)
    detections = get_contour_detections(motion_mask, ang, thresh=bbox_thresh)
    if len(detections) == 0:
        return np.array([])
    bboxes = detections[:, :4]
    scores = detections[:, -1]
    return non_max_suppression(bboxes, scores, threshold=nms_thresh)


def draw_bboxes(frame, detections):
    # Draws bounding boxes on the frame
    for det in detections:
        x1, y1, x2, y2 = det.astype(int)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)


def create_gif_from_images(save_path, image_path, ext):
    # Creates a GIF from images in a directory
    ext = ext.replace(".", "")
    image_paths = sorted(glob(os.path.join(image_path, f"*.{ext}")))
    image_paths.sort(key=lambda f: int("".join(filter(str.isdigit, f))))
    pil_images = [Image.open(p) for p in image_paths]
    pil_images[0].save(
        save_path, format="GIF",
        append_images=pil_images[1:],
        save_all=True, duration=50, loop=0
    )
    print(f"GIF gespeichert: {save_path}")


In [16]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")

fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {width}x{height} @ {fps:.1f} fps, {total} Frames")

motion_thresh_map = build_motion_thresh(height, width, MOTION_THRESH)
kernel = np.ones((7, 7), dtype=np.uint8)

ret, prev_bgr = cap.read()
if not ret:
    raise RuntimeError("First frame could not be read.")

frame_idx = 0
saved_idx = 0

while True:
    ret, curr_bgr = cap.read()
    if not ret:
        break
    frame_idx += 1

    if MAX_FRAMES and frame_idx > MAX_FRAMES:
        break
    if frame_idx % FRAME_SKIP != 0:
        continue

    detections = get_detections(
        prev_bgr, curr_bgr,
        motion_thresh=motion_thresh_map,
        bbox_thresh=BBOX_THRESH,
        nms_thresh=NMS_THRESH,
        mask_kernel=kernel
    )

    display = curr_bgr.copy()
    if len(detections) > 0:
        draw_bboxes(display, detections)

    out_path = os.path.join(OUTPUT_DIR, f"frame_{saved_idx:05d}.png")
    cv2.imwrite(out_path, display)
    saved_idx += 1

    prev_bgr = curr_bgr

cap.release()
print(f"{saved_idx} Frames processed.")

Video: 576x1024 @ 30.0 fps, 2591 Frames
2590 Frames processed.


In [ ]:
# GIF
create_gif_from_images(GIF_PATH, OUTPUT_DIR, ".png")

# MP4
image_paths = sorted(glob(os.path.join(OUTPUT_DIR, "*.png")))
image_paths.sort(key=lambda f: int("".join(filter(str.isdigit, f))))
if image_paths:
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out_fps = fps / max(FRAME_SKIP, 1)
    writer = cv2.VideoWriter(MP4_PATH, fourcc, out_fps, (width, height))
    for p in image_paths:
        writer.write(cv2.imread(p))
    writer.release()
    print(f"MP4 saved: {MP4_PATH}")

GIF gespeichert: output_optflow/optical_flow_speech.gif
MP4 gespeichert: output_optflow/optical_flow_speech.mp4


Compare the optical flow result with the frame differencing result.
- What does optical flow detect better?
- What does frame differencing detect better?

---

## Background Subtraction

The third method learns the background over many frames and marks everything that deviates from the learned background as motion. This is more robust than frame differencing because short brightness fluctuations are not immediately counted as motion.

In [ ]:
OUTPUT_DIR_BS = "output_bgsub"
GIF_PATH_BS   = "output_bgsub/bgsub_speech.gif"
MP4_PATH_BS   = "output_bgsub/bgsub_speech.mp4"

SUB_TYPE   = "MOG2"   # "MOG2" (Gaussian Mixture-based Background/Foreground Segmentation)
                      # or "KNN" (K-Nearest Neighbors based Background/Foreground Segmentation)
BBOX_THRESH_BS = 100  # kleiner als bei Optical Flow (100 statt 200):
                      # Background Subtraction ist präziser, kleinere Konturen sind verlässlich
NMS_THRESH_BS  = 1e-2

In [1]:
def get_motion_mask_bgsub(fg_mask, min_thresh=0, kernel=None):
# main function for background-subtraction detections
# takes backSub and single frame as input, returns NMS-filtered bounding boxes
# frame is passed as grayscale
    if kernel is None:
        kernel = np.ones((9, 9), dtype=np.uint8)

    _, thresh = cv2.threshold(fg_mask, min_thresh, 255, cv2.THRESH_BINARY)
    motion_mask = cv2.medianBlur(thresh, 3)
    motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    return motion_mask


In [ ]:
# Background Subtractor initialisation
kernel_bs = np.ones((9, 9), dtype=np.uint8)

if SUB_TYPE == "MOG2":
    backSub = cv2.createBackgroundSubtractorMOG2(varThreshold=16, detectShadows=True)
    backSub.setShadowThreshold(0.5)
else:
    backSub = cv2.createBackgroundSubtractorKNN(dist2Threshold=1000, detectShadows=True)


In [38]:
os.makedirs(OUTPUT_DIR_BS, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"Video nicht gefunden: {VIDEO_PATH}")

frame_idx = 0
saved_idx = 0

while True:
    ret, frame_bgr = cap.read()
    if not ret:
        break
    frame_idx += 1

    if MAX_FRAMES and frame_idx > MAX_FRAMES:
        break
    if frame_idx % FRAME_SKIP != 0:
        continue

    detections = get_detections_bgsub(
        backSub,
        cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY),
        bbox_thresh=BBOX_THRESH_BS,
        nms_thresh=NMS_THRESH_BS,
        kernel=kernel_bs
    )

    display = frame_bgr.copy()
    if len(detections) > 0:
        draw_bboxes(display, detections)

    out_path = os.path.join(OUTPUT_DIR_BS, f"frame_{saved_idx:05d}.png")
    cv2.imwrite(out_path, display)
    saved_idx += 1

cap.release()
print(f"{saved_idx} Frames verarbeitet.")

create_gif_from_images(GIF_PATH_BS, OUTPUT_DIR_BS, ".png")

image_paths_bs = sorted(glob(os.path.join(OUTPUT_DIR_BS, "*.png")))
image_paths_bs.sort(key=lambda f: int("".join(filter(str.isdigit, f))))
if image_paths_bs:
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out_fps = fps / max(FRAME_SKIP, 1)
    writer = cv2.VideoWriter(MP4_PATH_BS, fourcc, out_fps, (width, height))
    for p in image_paths_bs:
        writer.write(cv2.imread(p))
    writer.release()
    print(f"MP4 gespeichert: {MP4_PATH_BS}")

2591 Frames verarbeitet.
GIF gespeichert: output_bgsub/bgsub_speech.gif
MP4 gespeichert: output_bgsub/bgsub_speech.mp4
